In [1]:
import math

import torch
from llmfromscratch.llm_arch import GPTModel

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,    #1
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,       #2
    "qkv_bias": False
}

In [3]:
torch.manual_seed(123)

In [4]:
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trns_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feature

## 5.1 Evaluating Generative Text Models

In [5]:
import tiktoken
from llmfromscratch.llm_gen import generate_text_simplified

def text_to_token_ids(text, tokenizer):
    token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(token_ids).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flatten_text = token_ids.squeeze(0).tolist()
    decoded_text = tokenizer.decode(flatten_text)
    return decoded_text

In [6]:
start_text = "Every effort moves you closer to"
tokenizer = tiktoken.get_encoding("gpt2")
encoded_tensor = text_to_token_ids(start_text, tokenizer)
encoded_tensor

tensor([[6109, 3626, 6100,  345, 5699,  284]])

In [7]:
decoded_text =token_ids_to_text(encoded_tensor, tokenizer)
decoded_text

'Every effort moves you closer to'

In [8]:
generated_token_ids = generate_text_simplified(
                                model = model,
                                idx = encoded_tensor,
                                max_new_tokens = 100,
                                context_size = GPT_CONFIG_124M["context_length"]
                            )

In [9]:
#with CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
encoded_tensor = encoded_tensor.to(device)

In [10]:
#with CUDA
generated_token_ids = generate_text_simplified(
                                model = model,
                                idx = encoded_tensor,
                                max_new_tokens = 100,
                                context_size = GPT_CONFIG_124M["context_length"]
                            )

In [11]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107, 588, 11311]])  #  " really like chocolate"]

In [12]:
inputs = inputs.to(device)

with torch.no_grad():
    logits = model(inputs)
token_probas = torch.softmax(logits, dim=-1)


In [13]:
token_probas.shape

torch.Size([2, 3, 50257])

In [14]:
pred_token_ids = torch.argmax(token_probas, dim = -1, keepdim = True)
pred_token_ids

tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]], device='cuda:0')

In [15]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
      f" {token_ids_to_text(pred_token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


In [16]:
pred_token_ids[0].flatten().squeeze(0)

tensor([16657,   339, 42826], device='cuda:0')

In [17]:
token_probas.shape

torch.Size([2, 3, 50257])

In [18]:
text_idx = 0
target_probas_1 = token_probas[text_idx, [0,1,2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = token_probas[text_idx, [0,1,2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05], device='cuda:0')
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06], device='cuda:0')


In [19]:
target_probas = torch.cat((target_probas_1, target_probas_2))

In [20]:
target_probas

tensor([7.4540e-05, 3.1061e-05, 1.1563e-05, 1.0337e-05, 5.6776e-05, 4.7559e-06],
       device='cuda:0')

In [21]:
log_probas = torch.log(target_probas)
log_probas

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561],
       device='cuda:0')

In [22]:
avg_log_probas = torch.mean(log_probas, dim = -1)
avg_log_probas

tensor(-10.7940, device='cuda:0')

In [23]:
loss = (-1) * avg_log_probas

In [24]:
print("Logits shape:", token_probas.shape)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [25]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [26]:
loss_ce = torch.nn.functional.cross_entropy(logits_flat.to(device), targets_flat.to(device))
loss_ce

tensor(10.7940, device='cuda:0')

## 5.3 Calculating the train and evaluation set losses.

In [27]:
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

In [28]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Characters:", total_characters)
print("Tokens:", total_tokens)

Characters: 20479
Tokens: 5145


In [29]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [30]:
from llmfromscratch.llm_gen import create_dataloader_v1
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [31]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [32]:
def calculate_batch_loss(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    out_batch = model(input_batch)
    y_pred = out_batch.flatten(0,1)
    y = target_batch.flatten(0)
    loss = torch.nn.functional.cross_entropy(y_pred, y)

    return loss

def calculate_loss_loader(data_loader, model, device, num_batches = None):
    batch_num = 0
    loss = 0

    if num_batches is None:
       num_batches = len(data_loader)
    elif num_batches <= 0:
        return float("nan")

    print("A1")
    for x, y in data_loader:
        if batch_num < num_batches:
            batch_num += 1
            batch_loss = calculate_batch_loss(x, y, model, device)
            loss += batch_loss.item()
            print(f"Batch {batch_num} loss: {batch_loss.item()}")
        else:
            break

    loss /= batch_num

    return loss

In [33]:

batch = next(iter(val_loader))

In [34]:
with torch.no_grad():
    loader_loss = calculate_loss_loader(train_loader, model, device, 100)
loader_loss

A1
Batch 1 loss: 10.98016357421875
Batch 2 loss: 10.991239547729492
Batch 3 loss: 11.002304077148438
Batch 4 loss: 10.960235595703125
Batch 5 loss: 11.01591682434082
Batch 6 loss: 10.96568489074707
Batch 7 loss: 10.99540901184082
Batch 8 loss: 11.005659103393555
Batch 9 loss: 10.971637725830078


10.987583372328016

In [35]:
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)   #1
with torch.no_grad():                                        #2
    train_loss = calculate_loss_loader(train_loader, model, device)    #3
    val_loss = calculate_loss_loader(val_loader, model, device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

A1
Batch 1 loss: 11.014039039611816
Batch 2 loss: 11.013160705566406
Batch 3 loss: 10.993927001953125
Batch 4 loss: 10.968860626220703
Batch 5 loss: 10.998101234436035
Batch 6 loss: 10.96236801147461
Batch 7 loss: 10.938627243041992
Batch 8 loss: 10.99979019165039
Batch 9 loss: 10.999375343322754
A1
Batch 1 loss: 10.98110580444336
Training loss: 10.987583266364204
Validation loss: 10.98110580444336


In [36]:
logits.shape

torch.Size([2, 3, 50257])

In [37]:
logits[:3,:4,:5]

tensor([[[ 0.1113, -0.1057, -0.3666,  0.0229, -0.6258],
         [-0.6109, -0.5167, -0.7613,  0.3486,  0.2534],
         [ 0.5707, -0.6459, -0.0701, -0.4314, -0.1892]],

        [[-0.2968,  0.1949, -0.1649, -0.9598, -0.3897],
         [-0.8375,  0.0612, -0.4641,  0.0688,  0.0849],
         [ 0.5614,  0.6919,  0.8915, -0.6628, -1.3406]]], device='cuda:0')

## 5.4 Training LLM

In [38]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()  #1
    with torch.no_grad():                              #2
        train_loss = calculate_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calculate_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [43]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simplified(model=model, idx=encoded, max_new_tokens=50, context_size=context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))      #1
    model.train()

In [42]:
def train(model, train_lodaer, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []    #1
    tokens_seen, global_step = 0, -1

    for epoch_num in range(num_epochs):
        train_loss = 0
        val_loss = 0
        model.train()

        for batch_num, (input_batch, target_batch) in enumerate(train_loader):
            optimizer.zero_grad()
            batch_loss = calculate_batch_loss(input_batch, target_batch, model, device)
            #train_loss += batch_loss.item()
            batch_loss.backward()
            optimizer.step()

            global_step += 1
            tokens_seen += input_batch.shape[1]

            #train_loss /= (batch_num + 1)
            #print(f"Epoch {epoch_num} train loss: {train_loss}")

            if global_step % eval_freq == 0:    #6
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch_num+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

                #Printing sample text message after each batch
                #encoded_context = text_to_token_ids(start_context, tokenizer)
                #generate_text_simplified(model, encoded_context, 100, GPT_CONFIG_124M["context_length"])
                generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen


## 5.4.1 Enhancing training loop.

In [45]:
num_of_epoch = 15
min_lr = 1e-4
peak_lr = 0.01

In [46]:
total_batch_steps = len(train_loader) * num_of_epoch
warmup_steps =int(0.2 * total_batch_steps)
warmup_steps

27

In [47]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay = 0.1)


In [61]:
import math

def train_model_enhanced(model, train_lodaer, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer,
          initial_lr = 3e-5, min_lr = 1e-6):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []    #1
    tokens_seen, global_step = 0, -1
    total_batch_steps = len(train_loader) * num_epochs
    warmup_steps = int(0.2 * total_batch_steps)
    peak_lr = optimizer.param_groups[0]['lr']
    lr_increment = (peak_lr - initial_lr) / warmup_steps
    lr = 0

    for epoch_num in range(num_epochs):
        train_loss = 0
        val_loss = 0
        model.train()

        for batch_num, (input_batch, target_batch) in enumerate(train_loader):
            optimizer.zero_grad()
            batch_loss = calculate_batch_loss(input_batch, target_batch, model, device)
            #train_loss += batch_loss.item()

            if (global_step < warmup_steps):
            #Learning rate warm-up
                lr = initial_lr + global_step * lr_increment
            else:
            #Cosine lr decay
                progress = (global_step - warmup_steps) / (total_batch_steps - warmup_steps)
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            track_lrs.append(lr)
            batch_loss.backward()

            #Gradient Clipping
            if (global_step > warmup_steps):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            global_step += 1
            tokens_seen += input_batch.shape[1]

            #train_loss /= (batch_num + 1)
            #print(f"Epoch {epoch_num} train loss: {train_loss}")

            if global_step % eval_freq == 0:    #6
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch_num+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

                #Printing sample text message after each batch
                #encoded_context = text_to_token_ids(start_context, tokenizer)
                #generate_text_simplified(model, encoded_context, 100, GPT_CONFIG_124M["context_length"])
                generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [62]:
import tiktoken

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
peak_lr = 0.001
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)
tokenizer = tiktoken.get_encoding("gpt2")

n_epochs = 15
train_losses, val_losses, tokens_seen, lrs = train_model_enhanced(
    model, train_loader, val_loader, optimizer, device, num_epochs = n_epochs,
    eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=tokenizer,
    initial_lr=1e-5, min_lr=1e-5
)

A1
Batch 1 loss: 11.108301162719727
A1
Batch 1 loss: 11.094465255737305
Ep 1 (Step 000000): Train loss 11.108, Val loss 11.094
Every effort moves you rentingetic chief refusing holidays Shannon GamergateHay men methamphetamineDri ashamed Op EXT deductedObjectosticドラゴンWARNINGoking paw ostrchwitz Ruler RJ ,"Pokémonbind behav carries ENG triumph Middleow instrumental Adding Monroe Staten missedAmericans Kuala passengersinelli Discipline Ramsey Torn preferably Numbersstrip gain
A1
Batch 1 loss: 9.707655906677246
A1
Batch 1 loss: 9.848875045776367
Ep 1 (Step 000005): Train loss 9.708, Val loss 9.849
Every effort moves you,,,,,,, the,,,,,,,,,,, the,,,,,,,,, the,, the the                
A1
Batch 1 loss: 8.24348258972168
A1
Batch 1 loss: 8.577363014221191
Ep 2 (Step 000010): Train loss 8.243, Val loss 8.577
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
A1
Batch 1 loss: 6.623457908630371
A1
Batch 1 loss: 7.148375034332275
Ep 2 (Step 000015): Train loss 6.623, Val los